# IOAI — 2025 Selection 2 Baku Or Pasar (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-selection-2-baku-or-pasar'
for f in ['train.csv','test.csv','stopwords-ms.txt']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Baku or Pasar — MAIO 2025 (Selection 2)

말레이어 문장이 **정격(bahasa baku)** 인지 **구어체(bahasa pasar)** 인지 이진 분류한다.
원본은 class 0(baku) / 1–5(pasar) 지만 여기서는 `class = 0/1`(0=baku, 1=pasar) 로 이진화했다.

`data/train.csv`(`id,text,label`) 로 학습하고 `data/test.csv`(`id,text`) 를 예측해
`data/submission.csv`(`id,class`) 를 만든다. 채점은 held-out **F1**(pos=pasar).


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

train = pd.read_csv("data/train.csv")   # id, text, label
test  = pd.read_csv("data/test.csv")    # id, text
stopwords = [w.strip() for w in open("data/stopwords-ms.txt") if w.strip()]

vec = TfidfVectorizer(min_df=2, stop_words=stopwords)
Xtr = vec.fit_transform(train["text"])
Xte = vec.transform(test["text"])

clf = LogisticRegression(max_iter=2000)
clf.fit(Xtr, train["label"])
print("train F1:", round(f1_score(train["label"], clf.predict(Xtr)), 4))

pred = clf.predict(Xte)
pd.DataFrame({"id": test["id"], "class": pred.astype(int)}).to_csv("data/submission.csv", index=False)
print("saved data/submission.csv", len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['data/submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)